In [2]:
#Install
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas

#Usage
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52.52,
	"longitude": 13.41,
	"start_date": "2025-01-01",
	"end_date": "2026-09-30",
	"minutely_15": ["temperature_2m", "sunshine_duration", "wind_speed_80m", "wind_direction_80m", "is_day", "precipitation", "shortwave_radiation"],
	"timezone": "Europe/Berlin",
	"wind_speed_unit": "ms",
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process minutely_15 data. The order of variables needs to be the same as requested.
minutely_15 = response.Minutely15()
minutely_15_temperature_2m = minutely_15.Variables(0).ValuesAsNumpy()
minutely_15_sunshine_duration = minutely_15.Variables(1).ValuesAsNumpy()
minutely_15_wind_speed_80m = minutely_15.Variables(2).ValuesAsNumpy()
minutely_15_wind_direction_80m = minutely_15.Variables(3).ValuesAsNumpy()
minutely_15_is_day = minutely_15.Variables(4).ValuesAsNumpy()
minutely_15_precipitation = minutely_15.Variables(5).ValuesAsNumpy()
minutely_15_shortwave_radiation = minutely_15.Variables(6).ValuesAsNumpy()

minutely_15_data = {
    "date": pd.date_range(
        start=pd.to_datetime(minutely_15.Time(), unit="s", utc=True),
        end=pd.to_datetime(minutely_15.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=minutely_15.Interval()),
        inclusive="left"
    ).tz_convert(response.Timezone().decode()).tz_localize(None)
}

minutely_15_data["temperature_2m"] = minutely_15_temperature_2m
minutely_15_data["sunshine_duration"] = minutely_15_sunshine_duration
minutely_15_data["wind_speed_80m"] = minutely_15_wind_speed_80m
minutely_15_data["wind_direction_80m"] = minutely_15_wind_direction_80m
minutely_15_data["is_day"] = minutely_15_is_day
minutely_15_data["precipitation"] = minutely_15_precipitation
minutely_15_data["shortwave_radiation"] = minutely_15_shortwave_radiation

minutely_15_dataframe = pd.DataFrame(data = minutely_15_data)
print("\nMinutely15 data\n", minutely_15_dataframe)

# Save Excel
file_path = r"C:\Users\Dell\Desktop\Project\Weather.xlsx"

minutely_15_dataframe.to_excel(file_path, index=False)

# Confirm
import os
print("\nExcel saved successfully!")
print("File exists:", os.path.exists(file_path))
print("File path:", file_path)
print("File size:", os.path.getsize(file_path), "bytes")

Coordinates: 52.52000045776367°N 13.419998168945312°E
Elevation: 38.0 m asl
Timezone: b'Europe/Berlin'b'GMT+2'
Timezone difference to GMT+0: 7200s

Minutely15 data
                      date  temperature_2m  sunshine_duration  wind_speed_80m  \
0     2024-12-31 23:00:00        1.295500                0.0        9.402128   
1     2024-12-31 23:15:00        1.195500                0.0        9.481033   
2     2024-12-31 23:30:00        1.045500                0.0        9.560335   
3     2024-12-31 23:45:00        0.945500                0.0        9.580188   
4     2025-01-01 00:00:00        0.945500                0.0        9.660746   
...                   ...             ...                ...             ...   
61243 2026-09-30 22:45:00       17.995501                0.0        6.946222   
61244 2026-09-30 23:00:00       17.895500                0.0        6.815424   
61245 2026-09-30 23:15:00       17.795500                0.0        6.723094   
61246 2026-09-30 23:30:00       17.

In [12]:
import pandas as pd

# read weather excel file
weather = pd.read_excel(
    r"C:\Users\Dell\Desktop\Project\Weather.xlsx"
)

# convert date
weather["date"] = pd.to_datetime(
    weather["date"],
    errors="coerce"
)

# check datatype
print("Weather date datatype：")
print(weather["date"].dtype)

print("\nfailedconversion：")
print(weather["date"].isna().sum())

print("\nfirst5rows：")
print(weather[["date"]].head())

Weather date datatype：
datetime64[ns]

failedconversion：
0

first5rows：
                 date
0 2024-12-31 23:00:00
1 2024-12-31 23:15:00
2 2024-12-31 23:30:00
3 2024-12-31 23:45:00
4 2025-01-01 00:00:00
